In [ ]:
# user-001-key-1
# sk-hY02C1akTgjEgMVdb-IT3Q

# user-001-key-2
# sk-Q5NX5y59TbBGtn3zUYy3cw

# sk-hHqbkaGbNH_leZa3jmtgWg

API_KEY="sk-my-super-secret-key"

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key="sk-my-super-secret-key",
    base_url="http://localhost:4000"
)

response = client.chat.completions.create(
    model="glm-5.1",

    # ✅ EXACT Playground format
    messages=[
        {
            "role": "user",
            "content": "who are you?"
        },
        {
            "role": "assistant",
            "content": "I'm GLM, a large language model developed by Z.ai."
        },
        {
            "role": "user",
            "content": "What is current temperature of dhaka today?"
        }
    ],

    # ✅ MCP tools (same as Playground)
    tools=[
        {
            "type": "mcp",
            "server_label": "litellm",
            "server_url": "litellm_proxy/mcp/utility_server",
            "require_approval": "never"
        }
    ],
    tool_choice="auto",
    stream=True
)

# ✅ streaming handler (chat.completions style)
output_text = ""

for chunk in response:
    choice = chunk.choices[0]
    delta = choice.delta

    print("\n--- CHUNK ---")

    # Assistant role
    if delta.role:
        print(f"Role: {delta.role}")

    # Visible assistant text
    if delta.content:
        print(f"Content: {delta.content}")

    # Hidden reasoning
    if getattr(delta, "reasoning_content", None):
        print(f"Reasoning: {delta.reasoning_content}")

    # Tool calls
    if delta.tool_calls:
        for tool in delta.tool_calls:
            print("Tool Call:")
            print(f"  Name: {tool.function.name}")
            print(f"  Args: {tool.function.arguments}")

    # Provider-specific MCP info
    psf = getattr(delta, "provider_specific_fields", None)

    if psf:
        if "mcp_list_tools" in psf:
            print("Available MCP Tools:")
            for t in psf["mcp_list_tools"]:
                print(f"  - {t['function']['name']}")

        if "mcp_tool_calls" in psf:
            print("MCP Tool Calls:")
            for tc in psf["mcp_tool_calls"]:
                print(f"  - {tc['function']['name']}")
                print(f"    args: {tc['function']['arguments']}")

        if "mcp_call_results" in psf:
            print("MCP Results:")
            for r in psf["mcp_call_results"]:
                print(f"  - {r['name']}")
                print(f"    result: {r['result']}")

    # Finish reason
    if choice.finish_reason:
        print(f"Finish Reason: {choice.finish_reason}")

In [ ]:
from litellm import completion

response = await completion(
    model="openai/glm-5.1",
    api_base="http://localhost:4000",
    api_key="sk-my-super-secret-key",
    stream=True,

    messages=[
        {"role": "user", "content": "who are you?"},
        {"role": "assistant", "content": "I'm GLM, a large language model developed by Z.ai."},
        {"role": "user", "content": "Which medicine should I take for fever? Keep under 10 words."}
    ],
    tools=[
        {
            "type": "mcp",
            "server_label": "litellm",
            "server_url": "litellm_proxy/mcp/utility_server",
            "require_approval": "never"
        }
    ],
    tool_choice="auto"
)

full_text = ""

async for chunk in response:
    delta = chunk.choices[0].delta

    if getattr(delta, "content", None):
        print(delta.content, end="", flush=True)
        full_text += delta.content

print("\n\nFinal:", full_text)